# 01 · EDA — Rest-Mex 2025 (Pueblos Mágicos)

**Objetivo:** entender el dataset antes de construir los arquetipos (K-Means por pueblo → 10 vectores/pueblo) y la penalización de popularidad.

Preguntas que responde este notebook:
1. ¿Estructura, tipos y nulos del CSV? ¿Cuántos pueblos hay realmente (¿40?)
2. ¿Cómo se distribuyen `Polarity`, `Type` y `Region`?
3. ¿Qué tan desbalanceado está el número de reseñas por pueblo (hubness)? ¿Tulum es el máximo?
4. ¿Longitud de textos, truncados, mojibake, duplicados?
5. ¿Todos los pueblos tienen reseñas suficientes para K=10 clústeres?
6. ¿Qué vocabulario distingue a cada pueblo? (vista previa de los temas)

> **Nota de diseño:** la limpieza TF-IDF (minúsculas, sin acentos, sin stopwords) se usa **solo para análisis léxico aquí**. Para los embeddings (TourBERT/Gemini) conviene texto casi crudo: solo arreglar mojibake + NFKC + espacios.

In [ ]:
import re, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ftfy import fix_text
from ftfy.badness import badness
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.max_colwidth', 120, 'display.width', 160)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw' / 'Rest-Mex_2025_train.csv'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

K_CLUSTERS = 10          # arquetipos por pueblo
MIN_PER_CLUSTER = 5      # mínimo razonable de reseñas por clúster
print(ROOT, RAW.exists())

## 1. Carga y vista general

In [ ]:
try:
    df = pd.read_csv(RAW, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(RAW, encoding='latin-1')

print(df.shape)
df.info()
df.head()

In [ ]:
nulos = pd.DataFrame({'nulos': df.isna().sum(), '%': (df.isna().mean()*100).round(3)})
display(nulos)
print('Filas con Review vacío/espacios:', (df['Review'].fillna('').str.strip() == '').sum())
print('Filas con Title vacío/espacios :', (df['Title'].fillna('').str.strip() == '').sum())
print('Valores únicos:'); display(df[['Polarity','Town','Region','Type']].nunique())

## 2. Variables categóricas

In [ ]:
df['Town_clean'] = df['Town'].str.replace('_', ' ')
print('Pueblos únicos:', df['Town'].nunique(), '(esperado: 40)')
print('Polarity únicos:', sorted(df['Polarity'].dropna().unique()))
print('Types:', df['Type'].unique().tolist())

# ¿Cada pueblo pertenece a una sola región?
reg_por_pueblo = df.groupby('Town')['Region'].nunique()
print('Pueblos con >1 región:', (reg_por_pueblo > 1).sum())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
pol = df['Polarity'].value_counts().sort_index()
ax[0].bar(pol.index.astype(str), pol.values, color='steelblue'); ax[0].set_title('Polarity')
tp = df['Type'].value_counts()
ax[1].bar(tp.index, tp.values, color='darkorange'); ax[1].set_title('Type')
rg = df['Region'].value_counts()
ax[2].barh(rg.index[::-1], rg.values[::-1], color='seagreen'); ax[2].set_title('Region')
plt.tight_layout(); plt.show()
print((pol/pol.sum()*100).round(2).to_string())

## 3. Reseñas por pueblo (hubness)
Base para el notebook `04_penalty_tuning`: el castigo usará `log(n_pueblo)/log(n_max)`.

In [ ]:
town = (df.groupby('Town_clean')
          .agg(n=('Review','size'), pol_media=('Polarity','mean'),
               pct_neg=('Polarity', lambda s: (s <= 2).mean()*100),
               region=('Region','first'))
          .sort_values('n', ascending=False))
n_max = town['n'].max()
print('Pueblo más reseñado:', town['n'].idxmax(), n_max)
town['ratio_log'] = np.log1p(town['n']) / np.log1p(n_max)
town['ratio_lineal'] = town['n'] / n_max
display(town.round(3))

print('Top 5 concentran %.1f%% de las reseñas' % (town['n'].head(5).sum()/town['n'].sum()*100))
print('Mediana: %d | min: %d | max: %d | max/mediana: %.1fx' % (town['n'].median(), town['n'].min(), n_max, n_max/town['n'].median()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 9))
t = town.sort_values('n')
ax[0].barh(t.index, t['n'], color='steelblue'); ax[0].set_title('Reseñas por pueblo (lineal)')
ax[1].barh(t.index, t['n'], color='indianred'); ax[1].set_xscale('log'); ax[1].set_title('Reseñas por pueblo (log)')
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.scatter(town['ratio_lineal'], town['ratio_log'])
plt.xlabel('n / n_max'); plt.ylabel('log(n) / log(n_max)'); plt.title('Escala lineal vs logarítmica'); plt.show()

## 4. Pueblo × Type × Polarity

In [ ]:
ct = pd.crosstab(df['Town_clean'], df['Type'], normalize='index').loc[town.index]
fig, ax = plt.subplots(figsize=(6, 11))
im = ax.imshow(ct.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns)
ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index, fontsize=8)
ax.grid(False); plt.colorbar(im, label='proporción'); ax.set_title('Mezcla de Type por pueblo'); plt.show()

In [ ]:
print(pd.crosstab(df['Type'], df['Polarity'], normalize='index').round(3))
town[['pol_media','pct_neg']].sort_values('pol_media').head(8)

## 5. Calidad del texto
Mojibake, truncados (`...Más`, `�`), longitud y duplicados.

In [ ]:
df['Text_raw'] = df['Title'].fillna('') + '. ' + df['Review'].fillna('')
df['badness'] = df['Text_raw'].map(badness)
moji = df['badness'] > 0
print(f'Mojibake detectado: {moji.sum()} ({moji.mean()*100:.2f}%)')

df['Text_fix'] = df['Text_raw']
df.loc[moji, 'Text_fix'] = df.loc[moji, 'Text_raw'].map(fix_text)
df['badness_fix'] = df['Text_fix'].map(badness)
print('Sin arreglar tras fix_text:', (df['badness_fix'] > 0).sum())

ej = df[moji].head(4)
for a, b in zip(ej['Text_raw'], ej['Text_fix']):
    print('ANTES :', a[:130]); print('DESPUÉS:', b[:130]); print()

In [ ]:
df['n_chars'] = df['Text_fix'].str.len()
df['n_words'] = df['Text_fix'].str.split().str.len()
df['trunc_mas'] = df['Review'].fillna('').str.contains(r'\.\.\.\s*Más\s*$|…\s*Más\s*$', regex=True)
df['has_replacement'] = df['Text_fix'].str.contains('\ufffd')

print(df[['n_chars','n_words']].describe(percentiles=[.01,.05,.5,.95,.99]).round(1))
print('Truncadas por "Más":', df['trunc_mas'].sum(), '| con caracter � :', df['has_replacement'].sum())
print('Muy cortas (<20 chars):', (df['n_chars'] < 20).sum(), '| muy largas (>2000):', (df['n_chars'] > 2000).sum())

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(df['n_words'].clip(upper=df['n_words'].quantile(.99)), bins=60, color='steelblue')
ax[0].set_title('Palabras por reseña (recortado p99)')
df.boxplot(column='n_words', by='Polarity', ax=ax[1], showfliers=False)
ax[1].set_title('Palabras por Polarity'); plt.suptitle(''); plt.tight_layout(); plt.show()

In [ ]:
dup_exacto = df.duplicated(['Title','Review'], keep=False)
dup_fix = df.duplicated('Text_fix', keep=False)
print('Duplicados (Title,Review) [todas las copias]:', dup_exacto.sum())
print('Duplicados en Text_fix   [todas las copias]:', dup_fix.sum())
print('Filas sobrantes si se conserva la primera  :', df.duplicated('Text_fix', keep='first').sum())

# ¿Mismo texto con etiquetas distintas?
g = df[dup_fix].groupby('Text_fix').agg(towns=('Town','nunique'), pols=('Polarity','nunique'), types=('Type','nunique'))
print('Textos duplicados con >1 pueblo:', (g['towns'] > 1).sum(), '| >1 polaridad:', (g['pols'] > 1).sum())

> **Observación sobre tu `preprocessing.py`:** `df[~df['Text'].duplicated(keep=False)]` borra *todas* las copias, incluida la original. Para embeddings es mejor `keep='first'` (o `drop_duplicates`). Si hay textos idénticos en pueblos distintos, decide a mano a cuál asignarlos.

## 6. Texto "listo para embeddings" vs. texto TF-IDF
- **Embeddings:** `fix_text` → NFKC → espacios. Se conservan acentos, mayúsculas y puntuación.
- **TF-IDF (solo EDA):** minúsculas, sin acentos (conserva ñ), sin puntuación, sin stopwords (con tus excepciones de negación/intensidad).

In [ ]:
def normalize_embed(t):
    t = fix_text(t) if badness(t) > 0 else t
    t = unicodedata.normalize('NFKC', t)
    return re.sub(r'\s+', ' ', t).strip()

EXCEPCIONES = {'no','ni','nunca','jamas','tampoco','sin','muy','mucho','poco','nada','todo','bastante',
               'demasiado','mas','menos','tan','tanto','pero','aunque','sino','si'}

def clean_tfidf(t):
    t = t.lower().replace('ñ', '\ue000')
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    t = t.replace('\ue000', 'ñ')
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'[^a-z0-9ñ\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

SW_FILE = next((p for p in [ROOT/'stopwords-es.txt', ROOT/'data'/'stopwords-es.txt', ROOT/'data'/'raw'/'stopwords-es.txt'] if p.exists()), None)
if SW_FILE:
    sw = {clean_tfidf(l) for l in SW_FILE.read_text(encoding='utf-8').splitlines()} - EXCEPCIONES
else:
    sw = set('de la que el en y a los del se las por un para con una su al lo como mas pero sus le ya o este si porque esta entre cuando muy sin sobre tambien me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso ante ellos e esto mi antes algunos que unos yo otro otras otra el tanto esa estos mucho quienes nada muchos cual poco ella estar estas algunas algo nosotros es son fue era ser estaba estan'.split()) - EXCEPCIONES
    sw |= {'lugar','sitio','hotel','restaurante','tulum'}  # ajusta
print('Stopwords:', len(sw), '| archivo:', SW_FILE)

df['Text_embed'] = df['Text_raw'].map(normalize_embed)
df['Text_tfidf'] = df['Text_embed'].map(clean_tfidf).map(lambda t: ' '.join(w for w in t.split() if w not in sw))
df[['Text_embed','Text_tfidf']].sample(3, random_state=1)

## 7. Vocabulario global y distintivo por pueblo

In [ ]:
cnt = Counter(w for t in df['Text_tfidf'] for w in t.split())
top = pd.DataFrame(cnt.most_common(30), columns=['palabra','freq'])
plt.figure(figsize=(9,6)); plt.barh(top['palabra'][::-1], top['freq'][::-1], color='teal')
plt.title('Top 30 palabras (tras limpieza TF-IDF)'); plt.show()
print('Vocabulario:', len(cnt), '| hapax:', sum(v == 1 for v in cnt.values()))

In [ ]:
# 1 documento = todas las reseñas del pueblo -> términos más distintivos
docs = df.groupby('Town_clean')['Text_tfidf'].apply(' '.join)
vec = TfidfVectorizer(min_df=2, max_df=0.5, sublinear_tf=True)
X = vec.fit_transform(docs.values)
terms = np.array(vec.get_feature_names_out())
distintivos = {tw: ', '.join(terms[X[i].toarray().ravel().argsort()[::-1][:10]]) for i, tw in enumerate(docs.index)}
dist_df = pd.Series(distintivos, name='terminos_distintivos').to_frame().loc[town.index]
display(dist_df)

## 8. Viabilidad de K-Means (K=10) por pueblo
Se marca un pueblo como riesgoso si `n < K × MIN_PER_CLUSTER`.

In [ ]:
town['n_min_req'] = K_CLUSTERS * MIN_PER_CLUSTER
town['viable_k10'] = town['n'] >= town['n_min_req']
town['k_max_sugerido'] = (town['n'] // MIN_PER_CLUSTER).clip(upper=K_CLUSTERS).astype(int)
print('Pueblos con n < %d: %d' % (K_CLUSTERS*MIN_PER_CLUSTER, (~town['viable_k10']).sum()))
display(town.loc[~town['viable_k10'], ['n','k_max_sugerido']])
print('Reseñas por clúster promedio (K=10): mediana %.0f, mín %.0f' % ((town['n']/K_CLUSTERS).median(), (town['n']/K_CLUSTERS).min()))

## 9. Exportar resúmenes para los siguientes notebooks

In [ ]:
out = town.join(dist_df)
out.to_csv(PROC / 'eda_town_stats.csv', encoding='utf-8')
print('Guardado:', PROC / 'eda_town_stats.csv')

## 10. Hallazgos y decisiones (completar tras ejecutar)
- Nº de pueblos: ___ (¿coincide con 40?)
- Desbalance: máx/mediana = ___ ; ¿Tulum es el máximo? ___
- Mojibake / duplicados / truncados: ___
- Pueblos con K<10: ___ → decisión: reducir K / fusionar / dejar K=10
- Estrategia de texto para embeddings: `Text_embed` (título + reseña, sin limpieza agresiva)
- Ratio de penalización: lineal vs logarítmica → revisar en `04_penalty_tuning`
- ¿Usar `Polarity` para filtrar reseñas muy negativas antes de armar arquetipos? ___